<a href="https://colab.research.google.com/github/lucianotadeu/cnn_tensorFlow_video_opencv/blob/main/CNN_TensorFlow_Video_OpenCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**colab - Disciplina Processamento de imagem e sinais - FMU 2026.01**
*  Professor Luciano Tadeu Pereira
* email luciano.tadeu@fmu.br

---

**ATENÇÃO**: Antes de começar, faça uma cópia deste notebook para sua conta do Google Drive. Assim, você poderá editar e testar sem modificar o original.

Para isso, clique em "Arquivo" → "Salvar uma cópia no Drive".

OBS: Para executar este notebook, **não** é necessário um ambiente de execução com GPU.

---

# 🧠 CNN com TensorFlow & 🎥 Processamento de Vídeo com OpenCV
## Disciplina: Inteligência Artificial — Ensino Superior
### Continuação do Notebook: Processamento de Imagens e Sinais

---

> **⚡ Recomendação:** Ative a GPU no Google Colab antes de iniciar!  
> Menu: `Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)`

**Objetivo deste notebook:**  
Aprofundar o processamento de imagens utilizando **Redes Neurais Convolucionais (CNN)** com TensorFlow/Keras e explorar técnicas de **processamento de vídeo em tempo real** com OpenCV.

---

### 📚 Estrutura do Notebook

| Módulo | Aula | Tema |
|--------|------|------|
| **CNN** | Aula 1 | Fundamentos de Redes Neurais Convolucionais |
| **CNN** | Aula 2 | Construindo uma CNN do Zero (CIFAR-10) |
| **CNN** | Aula 3 | Data Augmentation e Regularização |
| **CNN** | Aula 4 | Transfer Learning com MobileNetV2 |
| **CNN** | Aula 5 | Visualizando o que a CNN aprende |
| **Vídeo** | Aula 6 | Fundamentos de Vídeo Digital |
| **Vídeo** | Aula 7 | Captura e Processamento Frame a Frame |
| **Vídeo** | Aula 8 | Detecção de Movimento e Subtração de Fundo |
| **Vídeo** | Aula 9 | Rastreamento de Objetos |
| **Vídeo** | Aula 10 | Pipeline Completo: CNN + Vídeo em Tempo Real |

---
# 🧠 MÓDULO 1 — Redes Neurais Convolucionais com TensorFlow

---
# 📘 AULA 1 — Fundamentos de Redes Neurais Convolucionais

## O que é uma CNN?

Uma **Rede Neural Convolucional** (Convolutional Neural Network) é um tipo especial de rede neural projetada para processar dados com estrutura de grade, como imagens.

### Por que não usar uma rede densa comum para imagens?
- Uma imagem 224×224×3 tem **150.528 valores de entrada**
- Uma camada densa conectaria **cada neurônio a todos os 150k valores** → explosão de parâmetros
- Redes densas **não aproveitam a localidade espacial** — pixels próximos são relacionados!

### Vantagens das CNNs
1. **Compartilhamento de pesos**: o mesmo filtro é aplicado em toda a imagem
2. **Invariância translacional**: detecta padrões independente da posição
3. **Hierarquia de features**: camadas iniciais aprendem bordas, camadas finais aprendem objetos complexos

### Arquitetura típica de uma CNN
```
INPUT → [CONV → ReLU → POOL] × N → FLATTEN → [DENSE → DROPOUT] × M → OUTPUT
```

| Camada | Função |
|--------|--------|
| **Conv2D** | Aplica filtros aprendíveis para extrair features |
| **ReLU** | Função de ativação não-linear: `max(0, x)` |
| **MaxPooling2D** | Reduz dimensionalidade, preserva features dominantes |
| **Flatten** | Transforma mapa 2D em vetor 1D |
| **Dense** | Classificação final com neurônios completamente conectados |
| **Dropout** | Regularização: desativa neurônios aleatoriamente durante treino |
| **Softmax** | Converte saída em probabilidades (soma = 1) |

In [ ]:
# ============================================================
# AULA 1 — Célula 1: Instalação e importações
# ============================================================

!pip install tensorflow opencv-python-headless matplotlib numpy scikit-learn --quiet

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tensorflow import keras
from tensorflow.keras import layers, models
import warnings
warnings.filterwarnings('ignore')

print("📦 Versões instaladas:")
print(f"  TensorFlow : {tf.__version__}")
print(f"  Keras      : {keras.__version__}")
print(f"  NumPy      : {np.__version__}")
print(f"  OpenCV     : {cv2.__version__}")

# Verificando GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU disponível: {gpus[0].name}")
    print("   O treinamento será significativamente mais rápido!")
else:
    print("\n⚠️  GPU não detectada. Usando CPU.")
    print("   Dica: Ative a GPU em 'Ambiente de execução → Alterar tipo de ambiente'")

In [ ]:
# ============================================================
# AULA 1 — Célula 2: Visualizando a operação de convolução
# ============================================================
# Antes de treinar redes neurais, vamos visualizar como
# a convolução funciona matematicamente com filtros conhecidos.

import urllib.request

# Create a Request object with a User-Agent header to avoid 403 Forbidden errors
req = urllib.request.Request(
    "https://upload.wikimedia.org/wikipedia/commons/4/47/PNG_transparency_demonstration_1.png",
    headers={'User-Agent': 'Mozilla/5.0'} # Common User-Agent header
)

# Open the URL with the request object and save the image
with urllib.request.urlopen(req) as response, open("img_demo.png", 'wb') as out_file:
    data = response.read() # Read the content
    out_file.write(data) # Write to file

img_demo = cv2.imread("img_demo.png")
img_gray = cv2.cvtColor(img_demo, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0

# Filtros clássicos que CNNs aprendem automaticamente
filtros = {
    'Detector de Bordas\n(Sobel X)': np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32),
    'Detector de Bordas\n(Sobel Y)': np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=np.float32),
    'Blur Gaussiano\n(Suavização)' : np.array([[1,2,1],[2,4,2],[1,2,1]], dtype=np.float32) / 16.0,
    'Detector de\nCantos (Laplace)': np.array([[0,-1,0],[-1,4,-1],[0,-1,0]], dtype=np.float32),
    'Realce de\nDetalhes'          : np.array([[-1,-1,-1],[-1,9,-1],[-1,-1,-1]], dtype=np.float32),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title('🖼️ Imagem Original\n(entrada da CNN)', fontsize=11, fontweight='bold')
axes[0].axis('off')

for ax, (nome, kernel) in zip(axes[1:], filtros.items()):
    resultado = cv2.filter2D(img_gray, -1, kernel)
    resultado_norm = np.clip(np.abs(resultado), 0, 1)
    ax.imshow(resultado_norm, cmap='hot')
    ax.set_title(f'Filtro: {nome}\nKernel {kernel.shape[0]}×{kernel.shape[1]}', fontsize=10)
    ax.axis('off')

plt.suptitle('🔬 Operação de Convolução — Filtros que CNNs aprendem automaticamente',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Insight fundamental:")
print("   Uma CNN aprende automaticamente os melhores filtros para a tarefa!")
print("   Não precisamos definir manualmente como nos exemplos acima.")
print("   O algoritmo de backpropagation ajusta os valores dos kernels durante o treino.")

In [ ]:
# ============================================================
# AULA 1 — Célula 3: Demonstração do MaxPooling
# ============================================================
# MaxPooling reduz as dimensões espaciais mantendo o valor máximo
# de cada janela. Isso torna a rede invariante a pequenas translações.

# Exemplo didático com matriz pequena
feature_map = np.array([
    [1,  3,  2,  4,  1,  0],
    [5,  6,  1,  2,  3,  1],
    [0,  2,  8,  3,  1,  4],
    [1,  3,  5,  7,  2,  1],
    [2,  4,  1,  0,  9,  3],
    [1,  0,  2,  1,  4,  6]
], dtype=float)

# MaxPooling 2×2 com stride=2
def max_pool_2x2(matrix):
    h, w = matrix.shape
    out = np.zeros((h//2, w//2))
    for i in range(0, h, 2):
        for j in range(0, w, 2):
            out[i//2, j//2] = np.max(matrix[i:i+2, j:j+2])
    return out

pooled = max_pool_2x2(feature_map)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = axes[0].imshow(feature_map, cmap='YlOrRd', vmin=0, vmax=9)
for i in range(6):
    for j in range(6):
        axes[0].text(j, i, str(int(feature_map[i,j])),
                    ha='center', va='center', fontsize=14, fontweight='bold')
# Desenhando grades do pooling
for k in range(0, 7, 2):
    axes[0].axhline(k-0.5, color='blue', linewidth=2)
    axes[0].axvline(k-0.5, color='blue', linewidth=2)
axes[0].set_title('Feature Map Original (6×6)\nLinhas azuis = janelas MaxPool 2×2', fontsize=11)
axes[0].axis('off')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(pooled, cmap='YlOrRd', vmin=0, vmax=9)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, str(int(pooled[i,j])),
                    ha='center', va='center', fontsize=18, fontweight='bold', color='darkblue')
axes[1].set_title('Após MaxPooling 2×2 (3×3)\n✅ Dimensão reduzida à metade!', fontsize=11)
axes[1].axis('off')
plt.colorbar(im2, ax=axes[1])

plt.suptitle('📉 MaxPooling — Redução de Dimensionalidade', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n📐 Dimensão antes do pooling : {feature_map.shape}")
print(f"📐 Dimensão após o pooling   : {pooled.shape}")
print(f"📉 Redução de parâmetros     : {feature_map.size} → {pooled.size} ({pooled.size/feature_map.size*100:.0f}% do original)")

---
# 📘 AULA 2 — Construindo uma CNN do Zero (CIFAR-10)

## Dataset CIFAR-10

O **CIFAR-10** é um dos benchmarks mais utilizados em visão computacional:
- **60.000 imagens** coloridas de 32×32 pixels
- **10 classes**: avião, carro, pássaro, gato, cervo, cachorro, sapo, cavalo, navio, caminhão
- **50.000 para treino** e **10.000 para teste**

## Arquitetura que vamos construir

```
Input (32×32×3)
    ↓
Conv2D(32, 3×3) + BatchNorm + ReLU  →  (32×32×32)
    ↓
Conv2D(32, 3×3) + BatchNorm + ReLU  →  (32×32×32)
    ↓
MaxPool(2×2) + Dropout(0.25)         →  (16×16×32)
    ↓
Conv2D(64, 3×3) + BatchNorm + ReLU  →  (16×16×64)
    ↓
Conv2D(64, 3×3) + BatchNorm + ReLU  →  (16×16×64)
    ↓
MaxPool(2×2) + Dropout(0.25)         →  (8×8×64)
    ↓
Flatten                               →  (4096)
    ↓
Dense(512) + ReLU + Dropout(0.5)     →  (512)
    ↓
Dense(10) + Softmax                  →  (10 classes)
```

In [ ]:
# ============================================================
# AULA 2 — Célula 1: Carregando e explorando o CIFAR-10
# ============================================================

from tensorflow.keras.datasets import cifar10

# Classes do CIFAR-10
CLASSES = ['Avião', 'Carro', 'Pássaro', 'Gato', 'Cervo',
           'Cachorro', 'Sapo', 'Cavalo', 'Navio', 'Caminhão']

# Carregando os dados (faz download automático na primeira execução)
print("📥 Carregando dataset CIFAR-10...")
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

print("\n📊 INFORMAÇÕES DO DATASET:")
print(f"  Treino  : {X_train.shape[0]:,} imagens {X_train.shape[1:]}")
print(f"  Teste   : {X_test.shape[0]:,} imagens {X_test.shape[1:]}")
print(f"  Classes : {len(CLASSES)} — {', '.join(CLASSES)}")
print(f"  Dtype   : {X_train.dtype} (valores de {X_train.min()} a {X_train.max()})")

# Visualizando exemplos de cada classe
fig, axes = plt.subplots(2, 10, figsize=(20, 5))
for classe in range(10):
    # Encontrando exemplos de cada classe
    indices = np.where(y_train.flatten() == classe)[0]
    for linha in range(2):
        axes[linha, classe].imshow(X_train[indices[linha]])
        axes[linha, classe].axis('off')
        if linha == 0:
            axes[linha, classe].set_title(CLASSES[classe], fontsize=9, fontweight='bold')

plt.suptitle('🖼️ Exemplos do Dataset CIFAR-10 (2 amostras por classe)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# AULA 2 — Célula 2: Pré-processamento dos dados
# ============================================================
# Pré-processamento é CRÍTICO para o desempenho da CNN:
# 1. Normalização: escalar pixels para [0,1] acelera convergência
# 2. One-Hot Encoding: converter rótulos para vetores binários

from tensorflow.keras.utils import to_categorical

# 1. Normalização: dividir por 255 para escalar [0,255] → [0,1]
# Redes neurais convergem melhor com entradas pequenas e padronizadas
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm  = X_test.astype('float32') / 255.0

# 2. One-Hot Encoding dos rótulos
# Antes: y_train = [3, 8, 8, 0, 6, ...] (índice da classe)
# Depois: [0,0,0,1,0,0,0,0,0,0] (vetor com 1 na posição da classe)
y_train_oh = to_categorical(y_train, num_classes=10)
y_test_oh  = to_categorical(y_test,  num_classes=10)

print("✅ Pré-processamento concluído:")
print(f"  X_train: {X_train.shape} → normalizado para [{X_train_norm.min():.1f}, {X_train_norm.max():.1f}]")
print(f"  y_train: {y_train.shape} → One-Hot {y_train_oh.shape}")
print(f"\n  Exemplo de rótulo antes : y_train[0] = {y_train[0][0]} ({CLASSES[y_train[0][0]]})")
print(f"  Exemplo de rótulo depois: y_train_oh[0] = {y_train_oh[0]}")

# Distribuição balanceada das classes
fig, ax = plt.subplots(figsize=(10, 4))
contagem = np.bincount(y_train.flatten())
bars = ax.bar(range(10), contagem, color=plt.cm.tab10(np.linspace(0, 1, 10)))
ax.set_xticks(range(10))
ax.set_xticklabels(CLASSES, rotation=30, ha='right')
ax.set_title('⚖️ Distribuição de Classes no Dataset de Treino', fontsize=12)
ax.set_ylabel('Quantidade de imagens')
for bar, val in zip(bars, contagem):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
            f'{val:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print("\n✅ Dataset perfeitamente balanceado — 5.000 imagens por classe.")

In [ ]:
# ============================================================
# AULA 2 — Célula 3: Construindo a arquitetura CNN
# ============================================================
# Usamos a API Sequential do Keras para empilhar camadas.
# BatchNormalization estabiliza o treino normalizando as ativações.

def criar_cnn(input_shape=(32, 32, 3), num_classes=10):
    """
    Cria a arquitetura CNN para classificação no CIFAR-10.

    Padrão de projeto: blocos CONV-CONV-POOL-DROPOUT
    Cada bloco aumenta os filtros e reduz o espaço espacial.
    """
    modelo = models.Sequential(name='CNN_CIFAR10')

    # ── Bloco 1: Extração de features de baixo nível (bordas, texturas) ──
    modelo.add(layers.Input(shape=input_shape))
    modelo.add(layers.Conv2D(32, (3,3), padding='same', activation='relu',
                             kernel_initializer='he_normal'))
    modelo.add(layers.BatchNormalization())  # Normaliza ativações → treino mais estável
    modelo.add(layers.Conv2D(32, (3,3), padding='same', activation='relu',
                             kernel_initializer='he_normal'))
    modelo.add(layers.BatchNormalization())
    modelo.add(layers.MaxPooling2D(pool_size=(2,2)))  # 32×32 → 16×16
    modelo.add(layers.Dropout(0.25))                 # Regularização

    # ── Bloco 2: Extração de features de médio nível (partes de objetos) ──
    modelo.add(layers.Conv2D(64, (3,3), padding='same', activation='relu',
                             kernel_initializer='he_normal'))
    modelo.add(layers.BatchNormalization())
    modelo.add(layers.Conv2D(64, (3,3), padding='same', activation='relu',
                             kernel_initializer='he_normal'))
    modelo.add(layers.BatchNormalization())
    modelo.add(layers.MaxPooling2D(pool_size=(2,2)))  # 16×16 → 8×8
    modelo.add(layers.Dropout(0.25))

    # ── Bloco 3: Features de alto nível (objetos completos) ──
    modelo.add(layers.Conv2D(128, (3,3), padding='same', activation='relu',
                             kernel_initializer='he_normal'))
    modelo.add(layers.BatchNormalization())
    modelo.add(layers.MaxPooling2D(pool_size=(2,2)))  # 8×8 → 4×4
    modelo.add(layers.Dropout(0.25))

    # ── Classificador: transforma features em probabilidades de classe ──
    modelo.add(layers.Flatten())                 # 4×4×128 = 2048 neurônios
    modelo.add(layers.Dense(512, activation='relu', kernel_initializer='he_normal'))
    modelo.add(layers.BatchNormalization())
    modelo.add(layers.Dropout(0.5))             # Dropout mais agressivo na parte densa
    modelo.add(layers.Dense(num_classes, activation='softmax'))  # 10 probabilidades

    return modelo

# Criando o modelo
modelo_cnn = criar_cnn()

# Compilando: definindo otimizador, função de perda e métricas
modelo_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',  # Padrão para classificação multiclasse
    metrics=['accuracy']
)

# Resumo da arquitetura
modelo_cnn.summary()

print("\n💡 Interpretando o summary:")
print("  'Output Shape' mostra as dimensões após cada camada")
print("  'Param #' mostra os pesos aprendíveis de cada camada")
total_params = modelo_cnn.count_params()
print(f"  Total de parâmetros: {total_params:,} ({total_params*4/1024/1024:.2f} MB em float32)")

In [ ]:
# ============================================================
# AULA 2 — Célula 4: Treinamento da CNN
# ============================================================
# Callbacks são funções executadas automaticamente durante o treino:
# - EarlyStopping: para o treino se não houver melhora
# - ReduceLROnPlateau: reduz learning rate quando a perda estagna
# - ModelCheckpoint: salva o melhor modelo automaticamente

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,          # Para se não melhorar por 10 épocas
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,           # Multiplica LR por 0.5
        patience=5,           # Aguarda 5 épocas sem melhora
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'melhor_modelo.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

print("🚀 Iniciando treinamento...")
print("   (Com GPU T4 demora ~2 min | Sem GPU pode demorar muito mais)\n")

historico = modelo_cnn.fit(
    X_train_norm, y_train_oh,
    batch_size=128,       # Número de imagens por atualização de pesos
    epochs=40,            # Máximo de épocas (EarlyStopping pode parar antes)
    validation_split=0.1, # 10% do treino para validação
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Treinamento concluído!")

In [ ]:
# ============================================================
# AULA 2 — Célula 5: Avaliação e curvas de aprendizado
# ============================================================

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Avaliação no conjunto de teste
loss_test, acc_test = modelo_cnn.evaluate(X_test_norm, y_test_oh, verbose=0)
print(f"📊 Resultado no conjunto de TESTE:")
print(f"   Acurácia : {acc_test*100:.2f}%")
print(f"   Perda    : {loss_test:.4f}")

# ── Gráfico 1: Curvas de Aprendizado ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epocas = range(1, len(historico.history['accuracy']) + 1)

axes[0].plot(epocas, historico.history['accuracy'],     'b-o', markersize=3, label='Treino')
axes[0].plot(epocas, historico.history['val_accuracy'], 'r-s', markersize=3, label='Validação')
axes[0].set_title('📈 Acurácia por Época', fontsize=12)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Acurácia')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])

axes[1].plot(epocas, historico.history['loss'],     'b-o', markersize=3, label='Treino')
axes[1].plot(epocas, historico.history['val_loss'], 'r-s', markersize=3, label='Validação')
axes[1].set_title('📉 Perda (Loss) por Época', fontsize=12)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss (Categorical Crossentropy)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('📊 Curvas de Aprendizado da CNN', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Gráfico 2: Matriz de Confusão ──
y_pred = np.argmax(modelo_cnn.predict(X_test_norm, verbose=0), axis=1)
y_true = y_test.flatten()
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_title('🎯 Matriz de Confusão — CIFAR-10', fontsize=13, fontweight='bold')
ax.set_xlabel('Predição', fontsize=11)
ax.set_ylabel('Rótulo Real', fontsize=11)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print("\n📋 Relatório de Classificação:")
print(classification_report(y_true, y_pred, target_names=CLASSES))

print("\n💡 Como interpretar a matriz de confusão:")
print("   Diagonal principal → acertos (quanto maior, melhor)")
print("   Fora da diagonal   → erros (qual classe foi confundida com qual)")

---
# 📘 AULA 3 — Data Augmentation e Regularização

## O problema do Overfitting

**Overfitting** ocorre quando o modelo aprende os dados de treino "de cor" mas não generaliza para novos dados. Sintomas:
- Acurácia de treino muito maior que a de validação
- Curva de loss de validação começa a subir enquanto a de treino cai

### Técnicas de Regularização
| Técnica | Mecanismo |
|---------|----------|
| **Dropout** | Desativa neurônios aleatoriamente durante treino |
| **BatchNorm** | Normaliza ativações, efeito regularizador indireto |
| **L2 (Weight Decay)** | Penaliza pesos grandes na função de perda |
| **Data Augmentation** | Gera variações artificiais das imagens de treino |
| **Early Stopping** | Para o treino antes do overfitting começar |

## Data Augmentation
Técnica de aumentar artificialmente o dataset aplicando **transformações aleatórias** nas imagens de treino — o modelo vê versões ligeiramente diferentes a cada época.

In [ ]:
# ============================================================
# AULA 3 — Célula 1: Visualizando Data Augmentation
# ============================================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Configurando o gerador de augmentação
datagen = ImageDataGenerator(
    rotation_range=15,       # Rotação aleatória ±15 graus
    width_shift_range=0.1,   # Deslocamento horizontal ±10%
    height_shift_range=0.1,  # Deslocamento vertical ±10%
    horizontal_flip=True,    # Espelhamento horizontal aleatório
    zoom_range=0.1,          # Zoom aleatório ±10%
    shear_range=0.1,         # Cisalhamento ±10%
    fill_mode='nearest'      # Preenchimento de pixels criados: repetir vizinho
)

# Selecionando uma imagem exemplo
idx_exemplo = np.where(y_train.flatten() == 1)[0][0]  # Primeiro carro
img_exemplo = X_train_norm[idx_exemplo:idx_exemplo+1]  # Shape: (1, 32, 32, 3)

# Gerando variações
fig, axes = plt.subplots(3, 8, figsize=(20, 8))

# Linha 0: imagem original repetida
for ax in axes[0]:
    ax.imshow(img_exemplo[0])
    ax.axis('off')
axes[0, 0].set_title('Original', fontsize=9)
axes[0, 3].set_title('← Sempre a mesma imagem →', fontsize=8, color='blue')

# Linhas 1 e 2: versões aumentadas
for linha in range(1, 3):
    for j, img_aug in enumerate(datagen.flow(img_exemplo, batch_size=1)):
        if j >= 8:
            break
        axes[linha, j].imshow(np.clip(img_aug[0], 0, 1))
        axes[linha, j].axis('off')

axes[1, 0].set_title('Augmented', fontsize=9)
axes[2, 0].set_title('Augmented', fontsize=9)

plt.suptitle('🔄 Data Augmentation — Mesma imagem, infinitas variações',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Benefícios do Data Augmentation:")
print("  ✅ Aumenta efetivamente o tamanho do dataset")
print("  ✅ Melhora a generalização para novos dados")
print("  ✅ Reduz overfitting sem coletar mais dados")
print("  ✅ Ensina o modelo a ser invariante a transformações")

---
# 📘 AULA 4 — Transfer Learning com MobileNetV2

## O que é Transfer Learning?

**Transfer Learning** (aprendizado por transferência) é a técnica de **reutilizar pesos** de uma rede pré-treinada em um grande dataset para resolver um novo problema.

### Por que usar Transfer Learning?
- Treinar CNNs profundas do zero requer **milhões de imagens** e **dias de computação**
- Modelos pré-treinados no ImageNet (1.2M imagens, 1000 classes) já aprenderam **features visuais universais**
- Com Transfer Learning, obtemos excelentes resultados com **poucos dados e minutos de treinamento**

### Estratégias
| Estratégia | Quando usar |
|------------|-------------|
| **Feature Extraction** | Dataset pequeno, similar ao original |
| **Fine-Tuning** | Dataset médio ou grande |
| **Treino do zero** | Dataset enorme e muito diferente |

### MobileNetV2
Arquitetura eficiente projetada para dispositivos móveis — excelente balanço entre precisão e velocidade.

In [ ]:
# ============================================================
# AULA 4 — Célula 1: Transfer Learning com MobileNetV2
# ============================================================

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# CIFAR-10 tem imagens 32×32, mas MobileNetV2 espera mínimo 96×96
# Precisamos redimensionar as imagens
IMG_SIZE = 96

def redimensionar_dataset(X, tamanho=IMG_SIZE):
    """Redimensiona batch de imagens para o tamanho esperado pelo modelo."""
    X_resized = np.array([cv2.resize(img, (tamanho, tamanho)) for img in X])
    return preprocess_input(X_resized * 255)  # MobileNetV2 usa [-1, 1]

print("🔄 Redimensionando imagens para MobileNetV2 (96×96)...")
# Usando subconjunto para demonstração mais rápida
N_DEMO = 5000
X_train_mb = redimensionar_dataset(X_train_norm[:N_DEMO])
X_test_mb  = redimensionar_dataset(X_test_norm[:1000])
y_train_mb = y_train_oh[:N_DEMO]
y_test_mb  = y_test_oh[:1000]
print(f"✅ Shapes: treino={X_train_mb.shape}, teste={X_test_mb.shape}")

# Carregando MobileNetV2 pré-treinado no ImageNet
# include_top=False: remove a cabeça de classificação original
# Mantemos apenas o 'corpo' extrator de features
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,     # Remove classificador do ImageNet
    weights='imagenet'     # Usa pesos pré-treinados
)

# CONGELANDO as camadas base — não atualizar esses pesos
# As features aprendidas no ImageNet já são poderosas
base_model.trainable = False

# Construindo o modelo com cabeça personalizada para CIFAR-10
entradas = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(entradas, training=False)
x = layers.GlobalAveragePooling2D()(x)  # Reduz mapa 3×3×1280 para vetor 1280
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
saidas = layers.Dense(10, activation='softmax')(x)

modelo_tl = keras.Model(entradas, saidas, name='TL_MobileNetV2')

modelo_tl.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

camadas_treinaveis = sum(1 for l in modelo_tl.layers if l.trainable)
camadas_congeladas = sum(1 for l in modelo_tl.layers if not l.trainable)
print(f"\n🏗️  Arquitetura Transfer Learning:")
print(f"   Camadas congeladas (pré-treinadas): {camadas_congeladas}")
print(f"   Camadas treináveis (novas)        : {camadas_treinaveis}")
print(f"   Parâmetros treináveis: {modelo_tl.count_params():,}")

# Treinando apenas a cabeça
print("\n🚀 Fase 1: Treinando apenas a cabeça de classificação...")
hist_tl = modelo_tl.fit(
    X_train_mb, y_train_mb,
    epochs=8,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

loss_tl, acc_tl = modelo_tl.evaluate(X_test_mb, y_test_mb, verbose=0)
print(f"\n📊 Resultado com Transfer Learning ({N_DEMO} amostras):")
print(f"   Acurácia: {acc_tl*100:.2f}%")
print("\n✅ Transfer Learning: alta performance com muito menos dados e tempo!")

---
# 📘 AULA 5 — Visualizando o que a CNN Aprende

## Interpretabilidade de CNNs

Uma crítica comum às redes neurais profundas é serem **"caixas pretas"**. Técnicas de visualização nos ajudam a entender o que a rede aprendeu:

- **Feature Maps**: mostram como cada filtro ativa para uma imagem específica
- **Filtros aprendidos**: visualiza os kernels aprendidos pela primeira camada
- **Grad-CAM**: destaca as regiões da imagem mais importantes para a decisão

In [ ]:
# ============================================================
# AULA 5 — Célula 1: Visualizando Feature Maps
# ============================================================
# Feature maps mostram como cada filtro da CNN 'vê' a imagem.
# Camadas iniciais detectam features simples (bordas, cores).
# Camadas profundas detectam features complexas (texturas, partes).

# Criando modelo que retorna saídas intermediárias
# Garante que o modelo está construído antes de tentar acessar suas entradas/saídas
if not modelo_cnn.built:
    modelo_cnn.build(input_shape=(None, 32, 32, 3)) # (None para o tamanho do batch)

nomes_camadas_conv = [l.name for l in modelo_cnn.layers if 'conv' in l.name]
modelo_visualizacao = keras.Model(
    inputs=modelo_cnn.layers[0].input,
    outputs=[modelo_cnn.get_layer(name).output for name in nomes_camadas_conv[:4]]
)

# Selecionando imagem de teste
img_teste = X_test_norm[2:3]  # Shape: (1, 32, 32, 3)
label_real = CLASSES[y_test[2][0]]

# Obtendo feature maps
feature_maps = modelo_visualizacao.predict(img_teste, verbose=0)

fig = plt.figure(figsize=(20, 14))

# Imagem original
ax_orig = fig.add_subplot(1, 5, 1)
ax_orig.imshow(img_teste[0])
ax_orig.set_title(f'Original\nClasse: {label_real}', fontsize=11, fontweight='bold')
ax_orig.axis('off')

# Feature maps das primeiras 4 camadas conv
for idx_camada, (fm, nome) in enumerate(zip(feature_maps, nomes_camadas_conv[:4])):
    n_filtros = min(fm.shape[-1], 8)  # Mostrar até 8 filtros
    # Calculando grid de subplots
    n_cols = n_filtros
    ax_title = fig.add_subplot(len(feature_maps)+1, 1, idx_camada+2)
    ax_title.axis('off')

fig2, axes2 = plt.subplots(4, 8, figsize=(20, 10))
for idx_camada, (fm, nome) in enumerate(zip(feature_maps, nomes_camadas_conv[:4])):
    for j in range(8):
        if j < fm.shape[-1]:
            axes2[idx_camada, j].imshow(fm[0, :, :, j], cmap='viridis')
        axes2[idx_camada, j].axis('off')
    axes2[idx_camada, 0].set_ylabel(f'{nome}\n{fm.shape[1]}×{fm.shape[2]}×{fm.shape[3]}',
                                      fontsize=9, rotation=0, ha='right', labelpad=80)

plt.suptitle(f'🔬 Feature Maps — O que a CNN "vê" em cada camada\nImagem: {label_real}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Observações:")
print("  Camadas iniciais  → features simples (bordas, gradientes de cor)")
print("  Camadas profundas → features abstratas (menos interpretáveis visualmente)")
print("  Filtros com ativação alta (amarelo) → regiões mais relevantes para essa feature")

In [ ]:
# ============================================================
# AULA 5 — Célula 2: Predições com visualização de confiança
# ============================================================

# Fazendo predições em exemplos de teste
n_exemplos = 12
indices = np.random.choice(len(X_test_norm), n_exemplos, replace=False)
imgs_teste = X_test_norm[indices]
labels_reais = y_test[indices].flatten()

predicoes = modelo_cnn.predict(imgs_teste, verbose=0)
classes_pred = np.argmax(predicoes, axis=1)
confiancas = np.max(predicoes, axis=1)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, (ax, img, real, pred, conf) in enumerate(
        zip(axes, imgs_teste, labels_reais, classes_pred, confiancas)):
    ax.imshow(img)
    cor_titulo = 'green' if real == pred else 'red'
    simbolo = '✅' if real == pred else '❌'
    ax.set_title(
        f'{simbolo} Real: {CLASSES[real]}\nPred: {CLASSES[pred]} ({conf*100:.1f}%)',
        fontsize=9, color=cor_titulo, fontweight='bold'
    )
    ax.axis('off')

acertos = sum(1 for r, p in zip(labels_reais, classes_pred) if r == p)
plt.suptitle(f'🎯 Predições da CNN — {acertos}/{n_exemplos} acertos neste lote',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 🎥 MÓDULO 2 — Processamento de Vídeo em Tempo Real com OpenCV

---
# 📘 AULA 6 — Fundamentos de Vídeo Digital

## O que é um Vídeo Digital?

Um vídeo digital é uma **sequência temporal de imagens** (frames) capturadas e reproduzidas em alta velocidade, criando a ilusão de movimento.

### Conceitos Fundamentais

| Conceito | Definição | Valor típico |
|----------|-----------|-------------|
| **FPS** (Frames Per Second) | Quadros por segundo | 24, 30, 60 fps |
| **Resolução** | Dimensão em pixels | 1280×720 (HD), 1920×1080 (Full HD) |
| **Codec** | Algoritmo de compressão | H.264, H.265, VP9 |
| **Bitrate** | Taxa de dados por segundo | 1-50 Mbps |
| **Espaço de cor** | Representação das cores | RGB, YUV, HSV |

### Por que 24 fps parece fluido?
O olho humano percebe movimento contínuo acima de ~16 fps. Cinema usa 24 fps por questões históricas de custo de filme, enquanto jogos e esportes usam 60+ fps para maior fluidez.

### Processamento Frame a Frame
```python
while video.isOpened():
    ret, frame = video.read()      # Lê o próximo frame
    frame_processado = processar(frame)  # Aplica algoritmo
    mostrar(frame_processado)      # Exibe resultado
```

In [ ]:
# ============================================================
# AULA 6 — Célula 1: Criando e explorando um vídeo sintético
# ============================================================
# No Colab não temos webcam, então criamos um vídeo sintético
# para demonstrar os conceitos de processamento frame a frame.

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
import matplotlib.animation as animation
from matplotlib.patches import Circle, Rectangle

# Parâmetros do vídeo
LARGURA, ALTURA = 640, 480
FPS = 30
N_FRAMES = 90  # 3 segundos a 30fps

# Criando o vídeo com OpenCV VideoWriter
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter('video_sintetico.mp4', fourcc, FPS, (LARGURA, ALTURA))

frames_gerados = []

for frame_idx in range(N_FRAMES):
    # Criando frame com gradiente de fundo
    frame = np.zeros((ALTURA, LARGURA, 3), dtype=np.uint8)

    # Fundo com gradiente dinâmico
    t = frame_idx / N_FRAMES
    for y in range(ALTURA):
        intensidade = int(20 + 30 * np.sin(y / ALTURA * np.pi + t * 2 * np.pi))
        # Garante que a intensidade não seja negativa antes de atribuir a uint8
        intensidade_clamped = max(0, intensidade)
        frame[y, :] = [intensidade_clamped//2, intensidade_clamped//3, intensidade_clamped]

    # Bola em movimento circular
    angulo = 2 * np.pi * frame_idx / N_FRAMES
    cx = int(LARGURA//2 + 150 * np.cos(angulo))
    cy = int(ALTURA//2 + 100 * np.sin(angulo))
    raio_bola = 40
    cor_bola = (0, int(200 * (1 - t)), int(255 * t))  # Verde → Azul
    # Garante que os componentes de cor não sejam negativos
    cor_bola_clamped = (max(0, cor_bola[0]), max(0, cor_bola[1]), max(0, cor_bola[2]))
    cv2.circle(frame, (cx, cy), raio_bola, cor_bola_clamped, -1)
    cv2.circle(frame, (cx, cy), raio_bola, (255, 255, 255), 2)  # Borda branca

    # Retângulo pulsante
    tamanho = int(50 + 30 * np.sin(frame_idx * 0.2))
    x1_r = LARGURA//2 - tamanho
    y1_r = ALTURA//2 - tamanho
    cv2.rectangle(frame, (x1_r, y1_r), (x1_r+2*tamanho, y1_r+2*tamanho), (255, 100, 0), 3)

    # Texto com contador de frames
    cv2.putText(frame, f'Frame: {frame_idx:03d} / {N_FRAMES}',
               (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, f'FPS: {FPS} | {FPS}fps x {N_FRAMES}frames = {N_FRAMES/FPS:.1f}s',
               (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

    video_writer.write(frame)
    frames_gerados.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

video_writer.release()
print(f"✅ Vídeo sintético criado: {N_FRAMES} frames a {FPS}fps = {N_FRAMES/FPS:.1f}s")

# Visualizando frames amostrados
indices_amostra = [0, 15, 30, 45, 60, 75, 89]
fig, axes = plt.subplots(1, len(indices_amostra), figsize=(20, 4))
for ax, idx in zip(axes, indices_amostra):
    ax.imshow(frames_gerados[idx])
    ax.set_title(f'Frame {idx}\nt={idx/FPS:.2f}s', fontsize=9)
    ax.axis('off')

plt.suptitle('🎬 Frames amostrados do vídeo sintético', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 📘 AULA 7 — Captura e Processamento Frame a Frame

## Pipeline de Processamento

O processamento de vídeo segue um padrão consistente:

```
FONTE DE VÍDEO
    ↓
cv2.VideoCapture(arquivo / 0 para webcam)
    ↓
[LOOP] cap.read() → obtém frame atual
    ↓
PROCESSAMENTO (filtros, detecções, anotações)
    ↓
cv2.VideoWriter / exibição
    ↓
cap.release() → libera recursos
```

### Webcam no Colab
O Google Colab não permite acesso direto à webcam por segurança, mas podemos simular captura em tempo real com JavaScript ou usar vídeos salvos.

In [ ]:
# ============================================================
# AULA 7 — Célula 1: Processamento Frame a Frame
# ============================================================
# Aplicamos vários filtros ao mesmo tempo em cada frame,
# criando um vídeo multi-janela processado.

def processar_frame(frame_bgr):
    """
    Aplica múltiplos processamentos em um frame.
    Retorna dicionário com versões processadas.
    """
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    gray      = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

    return {
        'Original'       : frame_rgb,
        'Cinza'          : gray,
        'Blur Gaussiano' : cv2.GaussianBlur(frame_rgb, (15, 15), 0),
        'Canny'          : cv2.Canny(gray, 50, 150),
        'HSV'            : cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV),
        'Negativo'       : cv2.bitwise_not(frame_rgb),
    }

# Lendo o vídeo e processando frames
# O cv2.VideoCapture pode falhar em ler todos os frames em alguns ambientes (Colab).
# Usamos a lista `frames_gerados` que foi criada anteriormente e contém todos os 90 frames.
frames_para_analisar = []
for frame_rgb in frames_gerados: # frames_gerados contém frames em RGB
    frames_para_analisar.append(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)) # Converte para BGR

print(f"✅ {len(frames_para_analisar)} frames lidos do vídeo")

# Processando e visualizando 3 momentos do vídeo
frames_amostra = [frames_para_analisar[0],
                  frames_para_analisar[len(frames_para_analisar)//2],
                  frames_para_analisar[-1]]
momentos = ['Frame inicial (t=0s)', 'Frame central (t=1.5s)', 'Frame final (t=3s)']

for frame_bgr, momento in zip(frames_amostra, momentos):
    processados = processar_frame(frame_bgr)
    fig, axes = plt.subplots(1, 6, figsize=(22, 4))

    for ax, (titulo, img) in zip(axes, processados.items()):
        cmap = 'gray' if len(img.shape) == 2 else None
        ax.imshow(img, cmap=cmap)
        ax.set_title(titulo, fontsize=9, fontweight='bold')
        ax.axis('off')

    plt.suptitle(f'🎬 {momento} — 6 formas de processar o mesmo frame',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
# 📘 AULA 8 — Detecção de Movimento e Subtração de Fundo

## Fundamentos da Subtração de Fundo

**Background Subtraction** é uma técnica fundamental para detectar objetos em movimento em cenas com câmera estática.

### Abordagens

1. **Diferença de frames**: `|frame_atual - frame_anterior|` → simples mas ruidoso
2. **MOG2** (Gaussian Mixture Model): modela cada pixel como mistura de gaussianas → robusto a iluminação
3. **KNN** (K-Nearest Neighbors): baseado em densidade de pixels → excelente para sombras

### Aplicações Reais
- Sistemas de segurança e vigilância
- Contagem de pessoas/veículos
- Análise de tráfego
- Monitoramento industrial

In [ ]:
# ============================================================
# AULA 8 — Célula 1: Subtração de Fundo — Diferença de Frames
# ============================================================

def detectar_movimento_diferenca(frame_atual, frame_anterior, limiar=30):
    """
    Detecta movimento pela diferença absoluta entre frames consecutivos.

    Parâmetros:
        frame_atual   : frame corrente em BGR
        frame_anterior: frame anterior em BGR
        limiar        : threshold para considerar mudança como movimento

    Retorna:
        mascara_movimento: imagem binária onde pixels brancos = movimento
        frame_anotado    : frame com contornos de movimento desenhados
    """
    gray_atual    = cv2.cvtColor(frame_atual, cv2.COLOR_BGR2GRAY)
    gray_anterior = cv2.cvtColor(frame_anterior, cv2.COLOR_BGR2GRAY)

    # Diferença absoluta entre frames
    diferenca = cv2.absdiff(gray_anterior, gray_atual)

    # Aplicando limiar para criar máscara binária
    _, mascara = cv2.threshold(diferenca, limiar, 255, cv2.THRESH_BINARY)

    # Operações morfológicas para remover ruído
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mascara = cv2.dilate(mascara, kernel, iterations=2)    # Preenche lacunas
    mascara = cv2.erode(mascara, kernel, iterations=1)     # Remove pequenos ruídos

    # Encontrando contornos dos objetos em movimento
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    frame_anotado = cv2.cvtColor(frame_atual, cv2.COLOR_BGR2RGB).copy()
    area_total_movimento = 0

    for contorno in contornos:
        area = cv2.contourArea(contorno)
        if area > 500:  # Ignorar movimento muito pequeno (ruído)
            x, y, w, h = cv2.boundingRect(contorno)
            cv2.rectangle(frame_anotado, (x, y), (x+w, y+h), (255, 0, 0), 2)
            cv2.putText(frame_anotado, f'Mov {area:.0f}px²',
                       (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
            area_total_movimento += area

    return mascara, frame_anotado, area_total_movimento

# Aplicando em frames consecutivos do vídeo
n_frames = len(frames_para_analisar)
areas_movimento = []

for i in range(1, n_frames):
    _, _, area = detectar_movimento_diferenca(
        frames_para_analisar[i], frames_para_analisar[i-1]
    )
    areas_movimento.append(area)

# Visualizando resultado em 3 momentos com muito movimento
indices_movimento = [10, 30, 60]
fig, axes = plt.subplots(3, 3, figsize=(18, 12))

for row, idx in enumerate(indices_movimento):
    mascara, anotado, area = detectar_movimento_diferenca(
        frames_para_analisar[idx], frames_para_analisar[idx-1]
    )
    diferenca_vis = cv2.absdiff(
        cv2.cvtColor(frames_para_analisar[idx-1], cv2.COLOR_BGR2GRAY),
        cv2.cvtColor(frames_para_analisar[idx], cv2.COLOR_BGR2GRAY)
    )

    axes[row, 0].imshow(cv2.cvtColor(frames_para_analisar[idx], cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f'Frame {idx} (atual)', fontsize=10)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(diferenca_vis, cmap='hot')
    axes[row, 1].set_title('Diferença entre frames', fontsize=10)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(anotado)
    axes[row, 2].set_title(f'Movimento detectado\n({area:.0f} px²)', fontsize=10)
    axes[row, 2].axis('off')

plt.suptitle('🚶 Detecção de Movimento — Diferença de Frames', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# AULA 8 — Célula 2: MOG2 — Subtração de Fundo Adaptativa
# ============================================================
# MOG2 (Mixture of Gaussians) é muito mais robusto que a diferença
# simples de frames. Ele APRENDE o modelo de fundo ao longo do tempo
# e se adapta a mudanças de iluminação.
#
# Como funciona:
# - Cada pixel é modelado como mistura de K distribuições gaussianas
# - Gaussianas com alta frequência e baixa variância = fundo estável
# - Pixels que não se encaixam no modelo = foreground (movimento)

# Criando o subtrator de fundo MOG2
subtrator_mog2 = cv2.createBackgroundSubtractorMOG2(
    history=50,           # Número de frames para aprender o fundo
    varThreshold=40,      # Limiar de variância para considerar fundo
    detectShadows=True    # Detecta e marca sombras em cinza
)

# Criando o subtrator KNN (alternativa moderna)
subtrator_knn = cv2.createBackgroundSubtractorKNN(
    history=50,
    dist2Threshold=400,
    detectShadows=True
)

mascaras_mog2 = []
mascaras_knn  = []

# Processando todos os frames
for frame in frames_para_analisar:
    mascara_mog2 = subtrator_mog2.apply(frame)
    mascara_knn  = subtrator_knn.apply(frame)
    mascaras_mog2.append(mascara_mog2)
    mascaras_knn.append(mascara_knn)

# Visualizando comparação dos métodos
indices_vis = [5, 25, 55, 80]
fig, axes = plt.subplots(4, 4, figsize=(18, 14))

for row, idx in enumerate(indices_vis):
    frame_rgb = cv2.cvtColor(frames_para_analisar[idx], cv2.COLOR_BGR2RGB)

    axes[row, 0].imshow(frame_rgb)
    axes[row, 0].set_title(f'Frame {idx}\nOriginal', fontsize=9)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(mascaras_mog2[idx], cmap='gray')
    axes[row, 1].set_title('MOG2\n(branco=frente, cinza=sombra)', fontsize=9)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(mascaras_knn[idx], cmap='gray')
    axes[row, 2].set_title('KNN\n(adaptativo)', fontsize=9)
    axes[row, 2].axis('off')

    # Fundo aprendido pelo MOG2
    fundo = subtrator_mog2.getBackgroundImage()
    if fundo is not None:
        axes[row, 3].imshow(cv2.cvtColor(fundo, cv2.COLOR_BGR2RGB))
    else:
        axes[row, 3].imshow(np.zeros_like(frame_rgb))
    axes[row, 3].set_title('Fundo aprendido\npelo MOG2', fontsize=9)
    axes[row, 3].axis('off')

plt.suptitle('🎭 Subtração de Fundo: Diferença × MOG2 × KNN',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Comparação dos métodos:")
print("  Diferença de frames : Simples, rápido, sensível a ruído e iluminação")
print("  MOG2                : Adapta ao fundo, detecta sombras, robusto")
print("  KNN                 : Mais preciso que MOG2, excelente para cenas complexas")

---
# 📘 AULA 9 — Rastreamento de Objetos (Object Tracking)

## O que é Rastreamento?

**Rastreamento** (tracking) é a capacidade de **seguir um objeto específico** ao longo dos frames de um vídeo. É diferente de detecção — a detecção encontra objetos em cada frame independentemente, enquanto o rastreamento mantém a identidade do objeto ao longo do tempo.

### Desafios do Rastreamento
- Oclusão (objeto fica atrás de outro)
- Mudança de iluminação
- Mudança de aparência do objeto
- Movimento rápido (motion blur)
- Múltiplos objetos similares

### Algoritmos disponíveis no OpenCV
| Algoritmo | Características | Velocidade |
|-----------|-----------------|------------|
| **CSRT** | Alta precisão, lida com oclusão | Médio |
| **KCF** | Kernelized Correlation — rápido | Rápido |
| **MOSSE** | Correlação de Fourier — muito rápido | Muito Rápido |
| **MIL** | Múltiplas instâncias | Médio |

In [ ]:
# ============================================================
# AULA 9 — Célula 1: Rastreamento com CSRT e Optical Flow
# ============================================================

# ── Método 1: Rastreamento com OpenCV CSRT ──
tracker = cv2.TrackerCSRT_create()

# Frame inicial para inicializar o tracker
frame_inicial = frames_para_analisar[0].copy()

# Detectando automaticamente a bola no frame inicial
# (usando subtração de fundo para encontrar o objeto)
gray_init = cv2.cvtColor(frame_inicial, cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(gray_init, 100, 255, cv2.THRESH_BINARY)
contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Encontrando o maior contorno (a bola)
bbox_inicial = None
max_area = 0
for c in contours:
    area = cv2.contourArea(c)
    if area > max_area and area < 50000:
        x, y, w, h = cv2.boundingRect(c)
        if w > 20 and h > 20:  # Ignorar objetos muito pequenos
            max_area = area
            bbox_inicial = (x, y, w, h)

if bbox_inicial is None:
    # Bbox padrão se não encontrar
    bbox_inicial = (LARGURA//2 + 100, ALTURA//2 - 50, 80, 80)

# Inicializando o tracker com o frame e a bbox
ok = tracker.init(frame_inicial, bbox_inicial)

# Rastreando ao longo dos frames
trajeto = []  # Para guardar o caminho percorrido
frames_rastreados = []

for frame in frames_para_analisar:
    ok, bbox = tracker.update(frame)
    frame_vis = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).copy()

    if ok:
        x, y, w, h = [int(v) for v in bbox]
        centro = (x + w//2, y + h//2)
        trajeto.append(centro)

        # Desenhando bounding box
        cv2.rectangle(frame_vis, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.circle(frame_vis, centro, 5, (255, 0, 0), -1)

        # Desenhando trajeto
        for i in range(1, len(trajeto)):
            alpha = i / len(trajeto)
            cor = (int(255*alpha), int(100*(1-alpha)), 0)
            cv2.line(frame_vis, trajeto[i-1], trajeto[i], cor, 2)

        cv2.putText(frame_vis, 'RASTREANDO', (x, y-10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    else:
        cv2.putText(frame_vis, 'OBJETO PERDIDO', (20, 40),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    frames_rastreados.append(frame_vis)

# Visualizando resultado
indices_track = [0, 20, 40, 60, 80, 88]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, idx in zip(axes, indices_track):
    ax.imshow(frames_rastreados[idx])
    ax.set_title(f'Frame {idx} — t={idx/FPS:.1f}s', fontsize=10)
    ax.axis('off')

plt.suptitle('🎯 Rastreamento de Objeto com CSRT — Trajeto em Laranja/Vermelho',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n✅ Rastreamento concluído!")
print(f"   Total de pontos no trajeto: {len(trajeto)}")
if trajeto:
    print(f"   Posição inicial : {trajeto[0]}")
    print(f"   Posição final   : {trajeto[-1]}")

In [ ]:
# ============================================================
# AULA 9 — Célula 2: Optical Flow — Lucas-Kanade
# ============================================================
# Optical Flow estima o VETOR DE MOVIMENTO de cada pixel entre
# dois frames consecutivos. É fundamental em:
# - Estimação de velocidade de objetos
# - Estabilização de vídeo
# - Compressão de vídeo (codecs)
# - Ego-motion estimation (para carros autônomos)
#
# Lucas-Kanade: calcula optical flow esparso para pontos de interesse
# Farneback: calcula optical flow denso para todos os pixels

# ── Optical Flow Esparso (Lucas-Kanade) ──
frame_ref = cv2.cvtColor(frames_para_analisar[0], cv2.COLOR_BGR2GRAY)

# Detectando pontos de interesse para rastrear (cantos de Harris)
pontos_iniciais = cv2.goodFeaturesToTrack(
    frame_ref,
    maxCorners=50,
    qualityLevel=0.3,
    minDistance=15,
    blockSize=7
)

# Parâmetros do Lucas-Kanade
params_lk = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
)

# ── Optical Flow Denso (Farneback) ──
def calcular_optical_flow_denso(frame_ant, frame_atual):
    """Calcula optical flow denso e retorna visualização colorida."""
    gray_ant   = cv2.cvtColor(frame_ant, cv2.COLOR_BGR2GRAY)
    gray_atual = cv2.cvtColor(frame_atual, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(
        gray_ant, gray_atual, None,
        pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0
    )

    # Convertendo vetores de fluxo para representação HSV (cor = direção, brilho = magnitude)
    magnitude, angulo = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros_like(frame_ant)
    hsv[..., 0] = angulo * 180 / np.pi / 2   # Direção → matiz (cor)
    hsv[..., 1] = 255                          # Saturação máxima
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)  # Velocidade → brilho

    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB), flow

# Calculando para alguns pares de frames
fig, axes = plt.subplots(3, 3, figsize=(18, 12))

pares = [(0, 5), (30, 35), (60, 65)]
for row, (idx_ant, idx_atual) in enumerate(pares):
    frame_ant   = frames_para_analisar[idx_ant]
    frame_atual = frames_para_analisar[idx_atual]

    flow_vis, flow = calcular_optical_flow_denso(frame_ant, frame_atual)

    axes[row, 0].imshow(cv2.cvtColor(frame_ant, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f'Frame {idx_ant}', fontsize=10)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(cv2.cvtColor(frame_atual, cv2.COLOR_BGR2RGB))
    axes[row, 1].set_title(f'Frame {idx_atual}', fontsize=10)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(flow_vis)
    axes[row, 2].set_title('Optical Flow (Farneback)\nCor=Direção, Brilho=Velocidade', fontsize=9)
    axes[row, 2].axis('off')

plt.suptitle('🌊 Optical Flow Denso (Farneback) — Estimação de Movimento',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n🎨 Interpretando a visualização do Optical Flow:")
print("   Cor     → direção do movimento (vermelho=direita, ciano=esquerda, verde=baixo)")
print("   Brilho  → magnitude da velocidade (mais brilhante = move mais rápido)")
print("   Preto   → pixels estáticos (fundo sem movimento)")

---
# 📘 AULA 10 — Pipeline Completo: CNN + Vídeo em Tempo Real

## Integrando CNN ao Pipeline de Vídeo

Esta aula integra **tudo que aprendemos** num pipeline completo:

```
FRAME DE VÍDEO
    ↓
Pré-processamento (resize, normalização)
    ↓
CNN → Classificação / Detecção
    ↓
Pós-processamento (Non-Maximum Suppression, etc.)
    ↓
Anotação Visual (bounding boxes, labels, confiança)
    ↓
FRAME ANOTADO
```

### Desafio do Tempo Real
Para 30 fps, cada frame deve ser processado em menos de **33 ms**. CNNs leves como MobileNet processam em ~5ms com GPU.

In [ ]:
# ============================================================
# AULA 10 — Célula 1: Pipeline CNN + Vídeo Simulado
# ============================================================
# Criamos um pipeline que simula classificação em tempo real.
# Para cada frame, aplicamos detecção de movimento E classificação CNN.

import time

# Recriando o subtrator de fundo para análise limpa
subtrator = cv2.createBackgroundSubtractorMOG2(history=30, varThreshold=30, detectShadows=False)

# ── Simulando classificação de ROIs ──
# Em um sistema real, extrairíamos regiões detectadas e as classificaríamos com a CNN.
# Aqui simulamos o processo completo.

def pipeline_completo(frame_bgr, subtrator, modelo_cnn, frame_idx):
    """
    Pipeline completo: subtração de fundo + CNN para classificação.
    Em produção, a CNN classificaria cada região detectada.
    """
    tempo_inicio = time.time()
    frame_anotado = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB).copy()
    h, w = frame_bgr.shape[:2]

    # ── Passo 1: Subtração de Fundo ──
    mascara_fg = subtrator.apply(frame_bgr)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mascara_limpa = cv2.morphologyEx(mascara_fg, cv2.MORPH_OPEN, kernel)
    mascara_limpa = cv2.dilate(mascara_limpa, kernel, iterations=2)

    # ── Passo 2: Encontrando regiões de interesse ──
    contornos, _ = cv2.findContours(mascara_limpa, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    regioes_detectadas = []
    for contorno in contornos:
        area = cv2.contourArea(contorno)
        if area > 800:
            x, y, bw, bh = cv2.boundingRect(contorno)
            # Padding ao redor da região
            pad = 10
            x1 = max(0, x - pad)
            y1 = max(0, y - pad)
            x2 = min(w, x + bw + pad)
            y2 = min(h, y + bh + pad)
            regioes_detectadas.append((x1, y1, x2, y2, area))

    # ── Passo 3: CNN para cada região detectada ──
    for (x1, y1, x2, y2, area) in regioes_detectadas:
        roi = frame_bgr[y1:y2, x1:x2]
        if roi.size == 0 or roi.shape[0] < 4 or roi.shape[1] < 4:
            continue

        # Pré-processamento para a CNN (32×32, normalizado)
        roi_proc = cv2.resize(roi, (32, 32))
        roi_proc = cv2.cvtColor(roi_proc, cv2.COLOR_BGR2RGB).astype('float32') / 255.0
        roi_batch = np.expand_dims(roi_proc, axis=0)

        # Classificação
        predicao = modelo_cnn.predict(roi_batch, verbose=0)[0]
        classe_id = np.argmax(predicao)
        confianca = predicao[classe_id]
        label = CLASSES[classe_id]

        # Anotando no frame
        cor = (0, 200, 0) if confianca > 0.5 else (255, 165, 0)
        cv2.rectangle(frame_anotado, (x1, y1), (x2, y2), cor, 2)

        # Fundo do texto
        texto = f'{label}: {confianca*100:.0f}%'
        (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(frame_anotado, (x1, y1-th-8), (x1+tw+4, y1), cor, -1)
        cv2.putText(frame_anotado, texto, (x1+2, y1-4),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

    # ── Passo 4: HUD de informações ──
    tempo_ms = (time.time() - tempo_inicio) * 1000
    fps_estimado = 1000 / max(tempo_ms, 1)

    overlay = frame_anotado.copy()
    cv2.rectangle(overlay, (0, 0), (340, 80), (0, 0, 0), -1)
    frame_anotado = cv2.addWeighted(overlay, 0.5, frame_anotado, 0.5, 0)

    cv2.putText(frame_anotado, f'Frame: {frame_idx:03d}', (8, 22),
               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 1)
    cv2.putText(frame_anotado, f'Tempo CNN: {tempo_ms:.1f}ms | FPS est.: {fps_estimado:.0f}',
               (8, 44), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
    cv2.putText(frame_anotado, f'Objetos: {len(regioes_detectadas)} | Modelo: CNN_CIFAR10',
               (8, 66), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 255, 200), 1)

    return frame_anotado, mascara_limpa, tempo_ms

# Executando o pipeline em todos os frames
print("🚀 Executando pipeline CNN + Vídeo...")
frames_pipeline = []
tempos_processamento = []

for i, frame in enumerate(frames_para_analisar):
    resultado, mascara, tempo = pipeline_completo(frame, subtrator, modelo_cnn, i)
    frames_pipeline.append(resultado)
    tempos_processamento.append(tempo)

print(f"✅ Pipeline concluído!")
print(f"   Tempo médio por frame : {np.mean(tempos_processamento):.2f} ms")
print(f"   FPS estimado          : {1000/np.mean(tempos_processamento):.1f}")
print(f"   Tempo total           : {sum(tempos_processamento)/1000:.2f} s")

In [ ]:
# ============================================================
# AULA 10 — Célula 2: Visualização Final e Exportação
# ============================================================

# Visualizando frames do pipeline
indices_final = [0, 15, 30, 45, 60, 75]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, idx in zip(axes, indices_final):
    ax.imshow(frames_pipeline[idx])
    ax.set_title(f'Frame {idx} — t={idx/FPS:.1f}s', fontsize=10)
    ax.axis('off')

plt.suptitle('🎬 Pipeline Completo: CNN + Subtração de Fundo + Rastreamento',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Análise de performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(tempos_processamento, color='steelblue', linewidth=1)
axes[0].axhline(np.mean(tempos_processamento), color='red', linestyle='--',
               label=f'Média: {np.mean(tempos_processamento):.1f}ms')
axes[0].axhline(33.3, color='green', linestyle=':', label='Limite 30fps (33.3ms)')
axes[0].fill_between(range(len(tempos_processamento)), tempos_processamento,
                     alpha=0.3, color='steelblue')
axes[0].set_title('⏱️ Tempo de Processamento por Frame', fontsize=12)
axes[0].set_xlabel('Número do Frame')
axes[0].set_ylabel('Tempo (ms)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

fps_por_frame = [1000/max(t, 0.1) for t in tempos_processamento]
axes[1].hist(fps_por_frame, bins=20, color='seagreen', edgecolor='darkgreen', alpha=0.8)
axes[1].axvline(30, color='red', linestyle='--', label='30 FPS (tempo real)')
axes[1].set_title('📊 Distribuição de FPS Estimado', fontsize=12)
axes[1].set_xlabel('FPS estimado')
axes[1].set_ylabel('Frequência')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('📈 Análise de Performance do Pipeline', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Salvando vídeo processado
print("\n💾 Salvando vídeo processado...")
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_saida = cv2.VideoWriter('pipeline_resultado.mp4', fourcc, FPS, (LARGURA, ALTURA))

for frame_rgb in frames_pipeline:
    frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
    video_saida.write(frame_bgr)

video_saida.release()
print("✅ Vídeo salvo: 'pipeline_resultado.mp4'")
print("   Para baixar: from google.colab import files; files.download('pipeline_resultado.mp4')")


In [ ]:
# ============================================================
# AULA 10 — Célula 3: Usando a Webcam em Tempo Real
# ============================================================
# Esta célula mostra como usar a WEBCAM no Google Colab via JavaScript.
# Execute apenas se quiser capturar imagem da câmera.

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import PIL.Image
import io

def capturar_webcam():
    """
    Captura uma foto da webcam no Google Colab usando JavaScript.
    Retorna a imagem como array NumPy.
    """
    js_code = """
    async function capturarImagem() {
        const div = document.createElement('div');
        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // Aguarda 3 segundos para a câmera aquecer
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise(resolve => setTimeout(resolve, 3000));

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', 0.8);
    }
    capturarImagem();
    """
    try:
        dados_imagem = eval_js(js_code)
        binario = b64decode(dados_imagem.split(',')[1])
        pil_img = PIL.Image.open(io.BytesIO(binario))
        img_array = np.array(pil_img)
        return img_array
    except Exception as e:
        print(f"⚠️  Não foi possível acessar a webcam: {e}")
        print("   Isso é esperado em ambientes sem câmera.")
        return None

def classificar_com_cnn(imagem_rgb, modelo, classes):
    """Classifica uma imagem RGB usando a CNN treinada."""
    img_resized = cv2.resize(imagem_rgb, (32, 32))
    img_norm = img_resized.astype('float32') / 255.0
    img_batch = np.expand_dims(img_norm, axis=0)

    predicao = modelo.predict(img_batch, verbose=0)[0]
    top3_idx = np.argsort(predicao)[::-1][:3]

    print("\n🎯 Top-3 Classificações:")
    for i, idx in enumerate(top3_idx):
        barra = '█' * int(predicao[idx] * 20)
        print(f"  {i+1}. {classes[idx]:12s}: {predicao[idx]*100:5.1f}% {barra}")

    return predicao

print("📸 Captura de Webcam + Classificação CNN")
print("=" * 45)
print("Para capturar uma foto e classificar, descomente e execute:")
print()
print("# img = capturar_webcam()")
print("# if img is not None:")
print("#     plt.imshow(img)")
print("#     plt.title('Imagem capturada pela webcam')")
print("#     plt.axis('off')")
print("#     plt.show()")
print("#     predicao = classificar_com_cnn(img, modelo_cnn, CLASSES)")
print()
print("💡 Dica: O modelo foi treinado no CIFAR-10 (objetos comuns).")
print("   Aponte a câmera para um carro, animal ou objeto cotidiano.")

In [ ]:
# ============================================================
# AULA 10 — Célula Final: Resumo e Próximos Passos
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════════╗
║    RESUMO — CNN com TensorFlow & Processamento de Vídeo          ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  🧠 MÓDULO CNN:                                                  ║
║  ✅ Aula 1  - Convolução, MaxPooling, arquitetura CNN            ║
║  ✅ Aula 2  - Construindo e treinando CNN no CIFAR-10            ║
║  ✅ Aula 3  - Data Augmentation e regularização                  ║
║  ✅ Aula 4  - Transfer Learning com MobileNetV2                  ║
║  ✅ Aula 5  - Visualizando feature maps e predições              ║
║                                                                  ║
║  🎥 MÓDULO VÍDEO:                                               ║
║  ✅ Aula 6  - Conceitos de vídeo digital e criação de frames     ║
║  ✅ Aula 7  - Processamento frame a frame com OpenCV             ║
║  ✅ Aula 8  - Subtração de fundo: Diferença, MOG2, KNN           ║
║  ✅ Aula 9  - Rastreamento CSRT e Optical Flow Farneback         ║
║  ✅ Aula 10 - Pipeline completo CNN + Vídeo em tempo real        ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║  📚 Próximos tópicos sugeridos:                                  ║
║     → Segmentação semântica com U-Net e DeepLab                  ║
║     → Detecção de objetos com YOLOv8 em tempo real               ║
║     → Redes Generativas Adversariais (GANs)                      ║
║     → Transformers para visão: Vision Transformer (ViT)          ║
║     → Deploy de modelos: TensorFlow Lite para mobile             ║
║     → Pose Estimation com MediaPipe                              ║
║                                                                  ║
║  🔗 Recursos:                                                    ║
║     → tensorflow.org/tutorials/images                            ║
║     → docs.opencv.org/4.x/d9/df8/tutorial_root.html              ║
║     → keras.io/examples/vision                                   ║
║     → paperswithcode.com/methods/category/object-detection       ║
╚══════════════════════════════════════════════════════════════════╝
""")